# Phase 3e — Corpus expansion

The bridging phase between Phase 3 and Phase 4. It builds no model and adds no
features. It answers one question, **which seasons can this pipeline actually
use, and does it measure them correctly**, and it ends in a frozen manifest that
Phase 4 reads instead of rediscovering the corpus.

Phase 3 left the modelling phase underpowered. Style identification would have
clustered 21 fingerprints in 49 dimensions, and evolution analysis would have run
change-point detection on two seasons twelve years apart. Both needed corpus, and
the audit below establishes what corpus exists.

## Setup

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
if ".." not in sys.path:
    sys.path.insert(0, str(Path.cwd().parent.parent))

import pandas as pd
from scipy import stats

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

from src.config import FEATURES_DIR
from src.utils.constants import SEASONS
from src.extraction.catalog import audit, classify_coverage, missing_fixtures
from src.extraction.reconcile import reconcile_season
from src.validation.collection import facing_club_effect, generic_team_density
from src.validation.manifest import freeze_manifest

## 1. What StatsBomb open data actually holds

The Premier League scope did not survive the audit. Open data holds exactly
**two** PL seasons, 2003/04 and 2015/16, and both were already built. Manchester
United 2007/08, Manchester City 2022/23 and every other era-defining PL season
are absent.

So the scope widened from "the Premier League" to "seasons collected deeply
enough to support the margin object", which Phase 3d proved needs a per-match
opponent rather than a full league.

In [2]:
# The classifier is the thing that turns "does this season exist" into
# "can the pipeline use it". Coverage is computed from the match list, never
# read off a season label.
CANDIDATES = [("Premier League", "2003/2004"), ("Premier League", "2015/2016"),
              ("La Liga", "2015/2016"), ("Serie A", "2015/2016"),
              ("Ligue 1", "2015/2016"), ("1. Bundesliga", "2015/2016"),
              ("La Liga", "2008/2009"), ("Champions League", "2015/2016")]

from statsbombpy import sb
comps = sb.competitions()
rows = []
for comp_name, season_name in CANDIDATES:
    row = comps[(comps.competition_name == comp_name)
                & (comps.season_name == season_name)].iloc[0]
    result = audit(int(row.competition_id), int(row.season_id))
    rows.append({"competition": comp_name, "season": season_name,
                 "matches": result["n_matches"], "clubs": result["n_teams"],
                 "top_club": result["top_team"],
                 "top_club_matches": result["top_team_matches"],
                 "coverage": result["coverage"]})
pd.DataFrame(rows)

,competition,season,matches,clubs,top_club,top_club_matches,coverage
0,Premier League,2003/2004,38,20,Arsenal,38,single_club
1,Premier League,2015/2016,380,20,Chelsea,38,full
2,La Liga,2015/2016,380,20,Real Madrid,38,full
3,Serie A,2015/2016,380,20,Hellas Verona,38,full
4,Ligue 1,2015/2016,377,20,AS Monaco,38,full
5,1. Bundesliga,2015/2016,34,18,Bayer Leverkusen,34,single_club
6,La Liga,2008/2009,31,19,Barcelona,31,single_club
7,Champions League,2015/2016,1,2,Real Madrid,1,sparse


**Bundesliga 2015/16 is the row that matters.** It carries 18 club names and a
season label, and it is 34 Leverkusen matches. Building it would have repeated
the 2003/04 trap exactly, at the cost of a full extraction and feature run.
Champions League data is finals only, one match per season, so it cannot support
a season profile and is ruled out as a cross-check source.

## 2. Coverage is not usability

A season can hold a full schedule and still lack an event type the feature
pipeline reads. The dependency is declared once and a test keeps that declaration
honest against the feature source; here it is checked against the oldest
collections in scope, because that is where a missing type would actually appear.

In [3]:
from src.extraction.catalog import (PER_MATCH_EVENT_TYPES, REQUIRED_EVENT_TYPES,
                                    check_event_types)

print(f"{len(REQUIRED_EVENT_TYPES)} event types are required by src/features/")
print(f"{len(PER_MATCH_EVENT_TYPES)} of them appear in every match, so a sample "
      f"can be judged on those\n")

for label, cid, sid in [("La Liga 2004/05 (oldest that exists)", 11, 37),
                        ("La Liga 2007/08 (earliest we build)", 11, 24)]:
    report = check_event_types(cid, sid, sample_n=3)
    print(f"{label:<38} missing={report['missing']}  "
          f"passes/match={report['density'].get('Pass'):.0f}")

16 event types are required by src/features/
14 of them appear in every match, so a sample can be judged on those



La Liga 2004/05 (oldest that exists)   missing=[]  passes/match=858


La Liga 2007/08 (earliest we build)    missing=[]  passes/match=1176


Measured across 7 PL 2015/16 matches, fourteen required types appear in all
seven, `Tactical Shift` in five and `Own Goal For` in two. Checking a three-match
sample against all sixteen would have failed healthy seasons, so sampling judges
only the fourteen a single match is expected to carry.

## 3. The gate: does a built season agree with the league it claims to be?

The only fully independent check in the phase. A final table falls out of the
results, it is public, and nothing in this pipeline influenced it. FBref cannot
do this job, because its advanced statistics are StatsBomb-derived and agreement
would only prove the aggregation.

**One rule covers every coverage shape:** a club's points deficit against the
official table may not exceed 3 per match the collection is missing for that
club, and no club may hold *more* points than it finished with. The strictness
then falls out of the coverage instead of being configured.

In [4]:
rows = []
for season in SEASONS:
    report = reconcile_season(season)
    exact = int((report.deficit == 0).sum())
    rows.append({"season": season, "clubs": len(report), "exact": exact,
                 "within_bound": int(((report.deficit > 0) & report.ok).sum()),
                 "failed": int((~report.ok).sum()),
                 "verdict": "EXACT" if exact == len(report) else "explained"})
pd.DataFrame(rows).set_index("season")

,clubs,exact,within_bound,failed,verdict
season,,,,,
2003/2004,20,1,19,0,explained
2015/2016,20,20,0,0,EXACT
La Liga 2015/2016,20,20,0,0,EXACT
Serie A 2015/2016,20,20,0,0,EXACT
Ligue 1 2015/2016,20,17,3,0,explained
Barcelona 2007/2008,20,0,20,0,explained
Barcelona 2008/2009,20,0,20,0,explained
Barcelona 2009/2010,20,0,20,0,explained
Barcelona 2010/2011,20,0,20,0,explained


Sixty clubs across the three complete leagues reconcile with **zero deficit**.
Ligue 1 is 377 of 380 and its three affected clubs land at a deficit of 3 against
a maximum of 3, exactly at the limit rather than comfortably inside it, which is
what a correct build with three absent away wins looks like.

The bound is **one-sided on purpose**. A negative deficit is not a near miss: no
amount of missing data can manufacture points, so it would mean a double-counted
match, a season misattribution or a reversed scoreline.

In [5]:
# Which three fixtures Ligue 1 is missing, and whether they fall on one club.
from src.config import PROCESSED_DIR
ligue1 = pd.read_csv(PROCESSED_DIR / "matches_ligue12016.csv")
gaps = missing_fixtures(ligue1.rename(columns={"match_home_team": "home_team",
                                               "match_away_team": "away_team"}))
print("absent fixtures:")
for home, away in gaps:
    print(f"   {home}  v  {away}")
print(f"\nclubs involved: {len({c for pair in gaps for c in pair})} distinct, "
      f"one match each of 38")
print("every listed row is match_status == 'available', so these are missing "
      "upstream rather than failed extraction")

absent fixtures:
   Bastia  v  Gazélec Ajaccio
   Saint-Étienne  v  Paris Saint-Germain
   Troyes  v  Bordeaux

clubs involved: 6 distinct, one match each of 38
every listed row is match_status == 'available', so these are missing upstream rather than failed extraction


### Where "explained" is nearly vacuous, and must not be read as a pass

For a single-club season an opponent seen twice draws a ~108 point allowance
that no error could exceed. The teeth are on the **focal club**, which plays
every collected match, so its deficit is exact for the subset and the bound
closes to zero once the collection is complete.

In [6]:
FOCAL = {"2003/2004": "Arsenal",
         **{f"Barcelona {y-1}/{y}": "Barcelona" for y in range(2008, 2016)}}
rows = []
for season, club in FOCAL.items():
    r = reconcile_season(season).loc[club]
    rows.append({"season": season, "club": club, "played": int(r.played),
                 "points": int(r.points), "official": int(r.official_points),
                 "deficit": int(r.deficit), "max_deficit": int(r.max_deficit),
                 "complete": r.played == 38})
pd.DataFrame(rows).set_index("season")

,club,played,points,official,deficit,max_deficit,complete
season,,,,,,,
2003/2004,Arsenal,38,90,90,0,0,True
Barcelona 2007/2008,Barcelona,27,52,67,15,33,False
Barcelona 2008/2009,Barcelona,31,76,87,11,21,False
Barcelona 2009/2010,Barcelona,35,92,99,7,9,False
Barcelona 2010/2011,Barcelona,33,83,96,13,15,False
Barcelona 2011/2012,Barcelona,37,88,91,3,3,False
Barcelona 2012/2013,Barcelona,32,82,100,18,18,False
Barcelona 2013/2014,Barcelona,31,69,87,18,21,False
Barcelona 2014/2015,Barcelona,38,94,94,0,0,True


Two seasons are collected in full, and both reproduce their real points totals
with no slack at all: **Arsenal 2003/04 at 90 against an official 90**, and
**Barcelona 2014/15 at 94 against 94**. The Invincibles' points total falling out
of raw events exactly is the strongest single validation this project has.

## 4. Does the pipeline recover football that was already known?

The gate proves the results were ingested correctly. It says nothing about
whether the style features measure anything real.

Guardiola's Barcelona is the most documented side in football and the consensus
is specific. The directions below were **sealed in
`.scratch/corpus-expansion/prereg-08-face-validity.md` and committed one commit
before any number was computed**. That ordering is the evidence; a direction
chosen after seeing the number is a description, not a test.

In [7]:
from src.representation.era import team_contrast

PREDICTED = {"pass_completion_pct": "> 0", "avg_pass_length_m": "< 0",
             "pi_connectivity": "> 0", "poss_mean_directness": "< 0"}
features = list(PREDICTED)
rows = {}
for year in range(2009, 2013):
    style = pd.read_csv(FEATURES_DIR / f"match_team_style_barca{year}.csv")
    rows[f"{year-1}/{year}"] = team_contrast(style, "Barcelona", features)
margins = pd.DataFrame(rows).T
margins.loc["predicted"] = [PREDICTED[f] for f in features]
margins

,pass_completion_pct,avg_pass_length_m,pi_connectivity,poss_mean_directness
2008/2009,13.911752,-3.676374,0.160491,-0.155983
2009/2010,15.464503,-3.358521,0.173182,-0.17012
2010/2011,17.64906,-5.068885,0.177738,-0.18724
2011/2012,16.387817,-4.857004,0.176618,-0.174443
predicted,> 0,< 0,> 0,< 0


All sixteen required sign predictions hold, and the magnitudes are not marginal:
Barcelona completed **14 to 18 percentage points more** of its passes than the
sides it played, while passing 3.4 to 5.1 m shorter, with higher connectivity and
lower directness every season.

The control matters more than the result. An instrument that called every good
team patient and possession-heavy would be measuring **quality**, not style.

In [8]:
leicester = team_contrast(
    pd.read_csv(FEATURES_DIR / "match_team_style_2016.csv"),
    "Leicester City", ["poss_mean_directness"])["poss_mean_directness"]
barcelona = team_contrast(
    pd.read_csv(FEATURES_DIR / "match_team_style_barca2011.csv"),
    "Barcelona", ["poss_mean_directness"])["poss_mean_directness"]
print(f"Leicester 2015/16 directness margin : {leicester:+.3f}   (counter-attacking)")
print(f"Barcelona 2010/11 directness margin : {barcelona:+.3f}   (patient)")
print("\nsame feature, same instrument, opposite sides of zero")

Leicester 2015/16 directness margin : +0.082   (counter-attacking)
Barcelona 2010/11 directness margin : -0.187   (patient)

same feature, same instrument, opposite sides of zero


## 5. An outside opinion on xG

The one place a second *model* votes. Understat fits its own expected-goals model
on its own event data, so disagreement here is two models disagreeing rather than
an arithmetic error. Coverage starts at 2014/15.

In [9]:
from src.validation.understat import UNDERSTAT_SEASONS, compare_xg

rows = []
for season in UNDERSTAT_SEASONS:
    joined = compare_xg(season).dropna(subset=["understat_xg"])
    rows.append({"season": season, "rows": len(joined),
                 "goals_agree": int(joined.goals_agree.sum()),
                 "pearson_r": round(stats.pearsonr(joined.xg, joined.understat_xg)[0], 3),
                 "MAE": round((joined.xg - joined.understat_xg).abs().mean(), 3),
                 "bias": round(joined.xg.mean() - joined.understat_xg.mean(), 3)})
pd.DataFrame(rows).set_index("season")

,rows,goals_agree,pearson_r,MAE,bias
season,,,,,
2015/2016,760,760,0.911,0.238,-0.013
La Liga 2015/2016,760,760,0.933,0.251,-0.092
Serie A 2015/2016,760,760,0.932,0.209,-0.047
Ligue 1 2015/2016,754,754,0.906,0.236,-0.064
Barcelona 2014/2015,76,76,0.957,0.361,-0.272


Agreement runs from r = 0.906 to 0.957. That is what two competent models look
like; it does not make either of them right. What it would catch is a pipeline
that had quietly stopped measuring chances.

The join tolerates a day, because late kickoffs land on different calendar dates
in the two sources. **Scorelines agree on every joined row of every season**,
which is what makes the tolerance safe: a mispaired fixture would break goal
agreement immediately.

⚠️ **One finding worth carrying into the dashboard.** Barcelona 2014/15 carries a
bias of −0.272 where every other season sits near zero, and it lands on
**Barcelona itself (−0.497) rather than its opponents (−0.046)**. This is not a
pipeline error: our aggregation already reproduces StatsBomb's own per-shot xG
exactly on 1580 of 1580 rows, so it is two models disagreeing about a side
generating ~2.5 xG a game. **xG is not interchangeable across sources, and the
divergence grows with chance quality.** Anything displaying xG should name the
model behind it.

## 6. Is `collection_pass` measured, or just asserted?

Phase 3d established that drift belongs to the collection rather than the year.
The registry tags which seasons share a collection, and the tag decides something
consequential: same pass means comparable raw, across passes the paired contrast
is mandatory. Until now that tag was a hypothesis written into a dict.

Possessions per team is the measure, since that is what 3d caught drifting and it
is structural rather than stylistic. The focal club is excluded from single-club
seasons for the same reason `era.py` keeps it out of a drift estimate.

In [10]:
rows = []
for season, entry in SEASONS.items():
    rows.append({"season": season, "collection_pass": entry["collection_pass"],
                 "possessions_per_team": round(generic_team_density(season), 2)})
density = pd.DataFrame(rows)
density.groupby("collection_pass").possessions_per_team.agg(
    ["min", "max", "mean", "count"]).round(2)

,min,max,mean,count
collection_pass,,,,
big5_2015_16,95.08,100.01,98.65,4
messi_la_liga,85.95,95.37,90.25,8
retrospective_2003_04,108.00,108.00,108.00,1


### 2003/04 is a different pass — confirmed

108.0 against ~95 to 100 everywhere else. The confound runs **against** this
finding rather than for it: 2003/04's figure comes from Arsenal's opponents,
measured only while facing a possession-dominant side, which *suppresses*
possession counts. It reads highest anyway, so the real gap is wider than
measured.

### The Messi seasons are a different pass — not supported

The obvious test looked decisive and was wrong, which is worth showing rather
than hiding.

In [11]:
# 16 clubs appear in both Barcelona 2014/15 and La Liga 2015/16.
a = pd.read_csv(FEATURES_DIR / "match_team_style_barca2015.csv",
                usecols=["team", "poss_possessions"])
a = a[a.team != "Barcelona"].groupby("team").poss_possessions.mean()
b = pd.read_csv(FEATURES_DIR / "match_team_style_laliga2016.csv",
                usecols=["team", "poss_possessions"])
b = b[b.team != "Barcelona"].groupby("team").poss_possessions.mean()
both = sorted(set(a.index) & set(b.index))
gap = (b[both] - a[both])
print(f"clubs in both seasons                : {len(both)}")
print(f"clubs reading higher in 2015/16      : {int((gap > 0).sum())}/{len(gap)}")
print(f"mean gap                             : {gap.mean():+.1f} possessions")

effect = facing_club_effect("La Liga 2015/2016", "Barcelona")
print(f"\nbut inside ONE pass (La Liga 2015/16):")
print(f"  clubs facing Barcelona             : {effect['vs_club']:.1f}")
print(f"  the same clubs elsewhere           : {effect['vs_everyone_else']:.1f}")
print(f"  'played Barcelona' effect          : {effect['effect']:+.1f}")
print(f"\nresidual once corrected             : {gap.mean() + effect['effect']:+.1f}")

clubs in both seasons                : 16
clubs reading higher in 2015/16      : 16/16
mean gap                             : +13.7 possessions

but inside ONE pass (La Liga 2015/16):
  clubs facing Barcelona             : 84.8
  the same clubs elsewhere           : 100.9
  'played Barcelona' effect          : -16.1

residual once corrected             : -2.4


All 16 clubs read higher in the later season, mean +13.7 possessions, same league,
consecutive years. That reads as a collection boundary until you notice those
clubs appear in 2014/15 **only in their matches against Barcelona**. Measured
inside a single pass, facing Barcelona is worth −16.1 possessions, which is larger
than the gap it appeared to prove. The residual points the other way.

**The tag stays separate anyway, on cost asymmetry rather than evidence.**
Wrongly assuming "different pass" costs a contrast that was not needed; wrongly
assuming "same pass" lets a 3d-style artifact into a result.

### The four 2015/16 leagues — inconclusive, honestly

They agree within ~5%, far tighter than the 2003/04 gap, which is consistent with
one pass. But the Premier League sits ~4.7 possessions below the other three and
**no club plays in two of these leagues**, so the same-club test that settled the
La Liga question is unavailable here. Phase 4 should still prefer the contrast,
which costs little and removes the question.

## 7. The frozen manifest

One row per season, mixing what the registry declares with what the validators
measured. Phase 4 reads this rather than rediscovering the corpus.

In [12]:
manifest = freeze_manifest()
print(f"written to {FEATURES_DIR / 'phase3e_corpus_manifest.csv'}\n")
manifest[["season", "coverage", "role", "collection_pass", "matches", "clubs",
          "style_rows", "absent_fixtures", "possessions_per_team",
          "reconciles", "clubs_exact"]]

written to /Users/sothea/Documents/Files/Code/prem analysis/data/features/phase3e_corpus_manifest.csv



,season,coverage,role,collection_pass,matches,clubs,style_rows,absent_fixtures,possessions_per_team,reconciles,clubs_exact
0,2003/2004,single_club,series,retrospective_2003_04,38,20,76,NaN,108.00,True,1
1,2015/2016,full,field,big5_2015_16,380,20,760,0.0,95.08,True,20
2,La Liga 2015/2016,full,field,big5_2015_16,380,20,760,0.0,99.86,True,20
3,Serie A 2015/2016,full,field,big5_2015_16,380,20,760,0.0,100.01,True,20
4,Ligue 1 2015/2016,full,field,big5_2015_16,377,20,754,3.0,99.66,True,17
5,Barcelona 2007/2008,single_club,series,messi_la_liga,27,20,54,NaN,94.70,True,0
6,Barcelona 2008/2009,single_club,series,messi_la_liga,31,19,62,NaN,92.42,True,0
7,Barcelona 2009/2010,single_club,series,messi_la_liga,35,20,70,NaN,95.37,True,0
8,Barcelona 2010/2011,single_club,series,messi_la_liga,33,20,66,NaN,88.30,True,0
9,Barcelona 2011/2012,single_club,series,messi_la_liga,37,20,74,NaN,89.92,True,0


In [13]:
built = manifest[manifest.built]
field_clubs = int(built.loc[built.role == "field", "clubs"].sum())
series = int((built.role == "series").sum())
print(f"field club-seasons : {field_clubs}   (four 2015/16 leagues x 20 clubs)")
print(f"series seasons     : {series}   (8 Barcelona + Arsenal 2003/04)")
print(f"corpus total       : {field_clubs + series} club-seasons")
print(f"all reconcile      : {bool(built.reconciles.all())}")

field club-seasons : 80   (four 2015/16 leagues x 20 clubs)
series seasons     : 9   (8 Barcelona + Arsenal 2003/04)
corpus total       : 89 club-seasons
all reconcile      : True


## Where this leaves Phase 4

**The corpus is 89 club-seasons**, up from 21. Style identification has a
population to fit on and evolution analysis has a nine-season consecutive series
with known change points.

### ⚠️ The sampling constraint, which is binding

Barcelona is **9 of the 89**, and those nine sit far tighter together than nine
random clubs. At k=5 the expected cluster is ~18 points, so Barcelona alone is
half a cluster's mass and could manufacture an archetype that is really one club.

**Fit archetypes on the 80-club single-season field, where every point is a
distinct club, then _project_ the Barcelona series and Arsenal 2003/04 into that
fitted space.** Train on breadth, score depth. This needs no new machinery: it is
the `style_space` (fit) and `place()` (project) seam built in 3c and exercised in
3d. The `role` column in the manifest is what tells Phase 4 which is which.

Two secondary cautions. Barcelona won five of its eight seasons, so any "what
champions do" analysis skews to one club and must report **distinct clubs
alongside season counts**. And anything the Barcelona series says about football's
evolution is a statement about *Barcelona*, the same trap as 3d-i's
`league_moved`.

### Depth is uneven, and unevenly placed

2007/08 holds 27 matches against 2014/15's 38, a 40% difference in sample size,
and 2007/08 is the season carrying the entire "before Guardiola" comparison. It
is both the thinnest and one of the most load-bearing. The change-point claim
must state sample sizes rather than treating eight seasons as equivalent.

### What this phase did not establish

Nothing here tests the *style features* against an outside measurement. Face
validity tests their direction against tactical consensus, and the xG check tests
a quantity Phase 4 does not use. There is no second source for possession
segmentation, connectivity or directness, and there is unlikely to be one.